# 3DGS pipeline: images -> novel view video

Upload your multi-view photos as a Kaggle Dataset named `scene-images` (a zip or a folder is fine), then run this notebook cell by cell. It will: COLMAP SfM -> 3DGS train -> orbit render -> video.

In [ ]:
import os, sys, subprocess, json, shutil, glob
from pathlib import Path

ROOT = Path('/kaggle/working')
ROOT.mkdir(exist_ok=True)
print('ok')

## Step 1: system deps + COLMAP

In [ ]:
import subprocess
# colmap is NOT preinstalled; it needs apt. archive.ubuntu.com DNS is flaky,
# so bound every acquisition with a short timeout and few retries, and never
# let a failure hang the cell.
r = subprocess.run(['which', 'colmap'], capture_output=True)
already = r.returncode == 0
print('colmap already installed:', already)
if not already:
    import os
    os.environ['DEBIAN_FRONTEND'] = 'noninteractive'
    cmd = ('apt-get install -y -q colmap '
           '-o Acquire::http::Timeout=10 -o Acquire::https::Timeout=10 '
           '-o Acquire::Retries=0 > /dev/null 2>&1')
    subprocess.run(cmd, shell=True, timeout=180)
    r2 = subprocess.run(['which', 'colmap'], capture_output=True)
    print('colmap installed now:', r2.returncode == 0)


## Step 2: clone + build 3DGS

In [ ]:
%cd /kaggle/working
!git clone --recursive https://github.com/graphdeco-inria/gaussian-splatting.git 2>&1 | tail -5
%cd gaussian-splatting

# Kaggle T4 image has system CUDA 12.x (nvcc 12) + Python 3.12 + modern gcc.
# 3DGS official code wants CUDA SDK matching torch. Install torch cu121 (matches system CUDA 12).
!pip uninstall -y torch torchvision torchaudio > /dev/null 2>&1
!pip install torch==2.5.1 torchvision==0.20.1 --index-url https://download.pytorch.org/whl/cu121
!python -c "import torch, torchvision; print(torch.__version__, torch.version.cuda, torch.cuda.is_available())"
!nvcc --version | tail -1
!gcc --version | head -1

!pip install plyfile tqdm
!pip install --no-deps lpips

# T4 is compute capability 7.5; compile only that arch (faster, avoids arch-name issues).
import os
os.environ['TORCH_CUDA_ARCH_LIST'] = '7.5'

!pip install submodules/diff-gaussian-rasterization
!pip install submodules/simple-knn

!python -c "import simple_knn._C as k; import diff_gaussian_rasterization._C as g; print('simple_knn + diff_gaussian_rasterization OK')"
print('3DGS build done')


## Step 3: gather images into data/scene/images

Handles a dataset uploaded as a zip (Kaggle may keep the nested folder) or a flat folder of images.

In [ ]:
kaggle_input = Path('/kaggle/input')

# locate the scene-images dataset, whatever its path
candidates = []
for p in kaggle_input.glob('scene-images/**'):
    candidates.append(p)
for p in kaggle_input.glob('**/scene-images*'):
    candidates.append(p)

found = next((p for p in candidates if p.is_dir()), None)
if found is None:
    print('searching all of /kaggle/input for jpg/png...')
    found = kaggle_input
imgs = sorted([p for p in found.rglob('*') if p.suffix.lower() in ('.jpg','.jpeg','.png')])
print(f'found {len(imgs)} images under {found}')

images = ROOT / 'data' / 'scene' / 'images'
images.mkdir(parents=True, exist_ok=True)
for i, src in enumerate(imgs):
    shutil.copy2(src, images)
print(f'copied {len(imgs)} images -> {images}')

# Downscale every image to max 1600px (long edge) BEFORE COLMAP.
# Full 5712px CPU SIFT is extremely slow; 1600px keeps plenty of features.
from PIL import Image
MAX = 1600
for imgp in sorted(images.glob('*.jpg')) + sorted(images.glob('*.png')):
    with Image.open(imgp) as im:
        w, h = im.size
        scale = MAX / max(w, h)
        if scale < 1.0:
            im = im.resize((int(w*scale), int(h*scale)), Image.LANCZOS)
            im.save(imgp, quality=95)
print('downscaled to max 1600px')


## Step 4: COLMAP (features -> match -> SfM)

In [ ]:
%cd /kaggle/working
scene = ROOT / 'data' / 'scene'

# COLMAP ships a Qt/OpenGL GUI. In a headless container, run CPU-only SIFT
# (use_gpu 0) so it skips OpenGL/CUDA context creation entirely.
import os
os.environ['QT_QPA_PLATFORM'] = 'offscreen'
os.environ['DISPLAY'] = ''

# 2000 features is enough for 3DGS; sequential matcher is much faster
# than exhaustive for an ordered orbit sequence, and avoids RAM blowup.
!colmap feature_extractor \
  --database_path data/scene/database.db \
  --image_path data/scene/images \
  --ImageReader.camera_model SIMPLE_PINHOLE \
  --SiftExtraction.use_gpu 0 \
  --SiftExtraction.max_num_features 2000 \
  --SiftExtraction.max_image_size 1600 \
  --SiftExtraction.estimate_affine_shape 0 \
  --SiftExtraction.num_threads 2

!colmap sequential_matcher \
  --database_path data/scene/database.db \
  --SiftMatching.use_gpu 0 \
  --SiftMatching.num_threads 2 \
  --SequentialMatching.overlap 12 \
  --SequentialMatching.quadratic_overlap 1

!mkdir -p data/scene/sparse
!colmap mapper \
  --database_path data/scene/database.db \
  --image_path data/scene/images \
  --output_path data/scene/sparse


## Check: did COLMAP produce camera poses?

In [ ]:
from pathlib import Path
sp0 = ROOT / 'data' / 'scene' / 'sparse' / '0'
if (sp0 / 'images.bin').exists() or (sp0 / 'images.txt').exists():
    n = len(list(sp0.glob('images.*')))
    print('COLMAP SfM OK:', sp0)
else:
    print('COLMAP FAILED: no sparse/0 output. Check image overlap/texture.')
    print('sparse dir contents:', list((ROOT/'data'/'scene'/'sparse').rglob('*')))


## Step 5: train 3DGS

In [ ]:
%cd /kaggle/working/gaussian-splatting
!python train.py -s ../data/scene -m ../output/scene_run1 --iterations 7000 --test_iterations 7000 --save_iterations 7000 --eval


## Step 6: render orbit video (custom trajectory)

Uses render_custom.py (in this repo under scripts/). Copy it into the gaussian-splatting dir and run it; then ffmpeg joins the frames.

In [ ]:
%cd /kaggle/working/gaussian-splatting

# render_custom.py is inlined below (no extra input dataset needed)
import pathlib
pathlib.Path("render_custom.py").write_text(r"""
﻿# Custom orbit renderer for a trained 3DGS model.
#
# Run INSIDE the gaussian-splatting repo directory (so `from gaussian_renderer
# import render` and `from scene.gaussian_model import GaussianModel` resolve).
#
#   python render_custom.py
#     --model_path /kaggle/working/output/scene_run1
#     --output_dir /kaggle/working/output/orbit
#     --frames 120
#
# It reads the training cameras.json (written next to the trained model) only to
# recover image resolution + intrinsics + a rough scene center/scale. The orbit
# itself is procedural: N cameras circling the recovered center, looking at it.

import argparse
import json
import math
import os

import numpy as np
import torch
import torchvision

from gaussian_renderer import render
from scene.gaussian_model import GaussianModel


def focal_to_fov(focal, pixels):
    return 2 * math.atan(pixels / (2 * focal))


def look_at_view_matrix(eye, center, up):
    eye = np.asarray(eye, dtype=np.float64)
    center = np.asarray(center, dtype=np.float64)
    up = np.asarray(up, dtype=np.float64)
    z = eye - center
    z /= np.linalg.norm(z)
    x = np.cross(up, z)
    x /= np.linalg.norm(x)
    y = np.cross(z, x)
    y /= np.linalg.norm(y)
    R = np.stack([x, y, z], axis=1)  # world->cam rotation (cols)
    t = -R.T @ eye
    Rt = np.eye(4)
    Rt[:3, :3] = R.T
    Rt[:3, 3] = t
    return torch.tensor(Rt, dtype=torch.float32, device="cuda")


class MiniCam:
    def __init__(self, width, height, fovy, fovx, world_view_transform):
        self.image_width = width
        self.image_height = height
        self.FoVy = fovy
        self.FoVx = fovx
        self.znear = 0.01
        self.zfar = 100.0
        self.world_view_transform = world_view_transform
        self.projection_matrix = get_projection_matrix(self.znear, self.zfar, fovx, fovy).transpose(0, 1).cuda()
        self.full_proj_transform = self.world_view_transform.unsqueeze(0).bmm(self.projection_matrix.unsqueeze(0)).squeeze(0)
        self.camera_center = self.world_view_transform.inverse()[3, :3]


def get_projection_matrix(znear, zfar, fov_x, fov_y):
    tan_half_y = math.tan(fov_y / 2)
    tan_half_x = math.tan(fov_x / 2)
    top = tan_half_y * znear
    bottom = -top
    right = tan_half_x * znear
    left = -right
    P = torch.zeros(4, 4)
    z_sign = 1.0
    P[0, 0] = 2.0 * znear / (right - left)
    P[1, 1] = 2.0 * znear / (top - bottom)
    P[0, 2] = (right + left) / (right - left)
    P[1, 2] = (top + bottom) / (top - bottom)
    P[3, 2] = z_sign
    P[2, 2] = z_sign * zfar / (zfar - znear)
    P[2, 3] = -(zfar * znear) / (zfar - znear)
    return P


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model_path", required=True)
    ap.add_argument("--output_dir", required=True)
    ap.add_argument("--frames", type=int, default=120)
    ap.add_argument("--radius_scale", type=float, default=1.0,
                    help="orbit radius as a multiple of the recovered scene radius")
    ap.add_argument("--height_scale", type=float, default=0.3,
                    help="camera height as a multiple of the recovered scene radius")
    args = ap.parse_args()

    model_path = args.model_path
    ply = os.path.join(model_path, "point_cloud", "iteration_7000", "point_cloud.ply")
    if not os.path.exists(ply):
        # fall back to whatever iteration exists
        pc_root = os.path.join(model_path, "point_cloud")
        its = [d for d in sorted(os.listdir(pc_root)) if d.startswith("iteration_")]
        if not its:
            raise SystemExit(f"no trained point cloud under {pc_root}")
        ply = os.path.join(pc_root, its[-1], "point_cloud.ply")
        print(f"using {ply}")

    gaussians = GaussianModel(sh_degree=3)
    gaussians.load_ply(ply)

    # recover intrinsics + scene center from training cameras.json
    cams_json = os.path.join(model_path, "cameras.json")
    if not os.path.exists(cams_json):
        raise SystemExit(f"no {cams_json}; run training first")
    with open(cams_json) as f:
        cams = json.load(f)

    first = cams[0]
    w, h = int(first["width"]), int(first["height"])
    fx, fy = first["fx"], first["fy"]
    fovx = focal_to_fov(fx, w)
    fovy = focal_to_fov(fy, h)

    positions = np.array([c["position"] for c in cams])
    center = positions.mean(axis=0)
    radius = float(np.linalg.norm(positions - center, axis=1).max())
    if radius <= 0:
        radius = 1.0
    print(f"scene center={center}, radius={radius}, res={w}x{h}")

    R = radius * args.radius_scale
    H = radius * args.height_scale

    os.makedirs(args.output_dir, exist_ok=True)
    bg = torch.tensor([0, 0, 0], dtype=torch.float32, device="cuda")

    # a PipelineParams-like minimal object; defaults match train.py
    class Pipe:
        debug = False
        antialiasing = False
        convert_SHs_python = False
        compute_cov3D_python = False
    pipe = Pipe()

    n = args.frames
    with torch.no_grad():
        for i in range(n):
            a = 2 * math.pi * i / n
            eye = np.array([center[0] + R * math.cos(a),
                            center[1] + R * math.sin(a),
                            center[2] + H])
            view = look_at_view_matrix(eye, center, [0, 0, 1])
            cam = MiniCam(w, h, fovy, fovx, view)
            out = render(cam, gaussians, pipe, bg)
            img = out["render"].clamp(0, 1)
            torchvision.utils.save_image(img, os.path.join(args.output_dir, f"{i:05d}.png"))
            if (i + 1) % 20 == 0:
                print(f"rendered {i + 1}/{n}")

    print(f"done -> {args.output_dir}")


if __name__ == "__main__":
    main()
""")

!python render_custom.py --model_path ../output/scene_run1 --output_dir ../output/orbit --frames 120
!ffmpeg -y -framerate 24 -pattern_type glob -i "../output/orbit/*.png" -c:v libx264 -pix_fmt yuv420p ../output/novel_view.mp4 2>&1 | tail -3
from pathlib import Path
v = Path('/kaggle/working/output/novel_view.mp4')
print('video:', v, 'size:', v.stat().st_size if v.exists() else 'MISSING')


## Step 7: metrics (PSNR on held-out COLMAP views)

Renders the COLMAP test split with the official render.py, then compares to ground truth.

In [ ]:
%cd /kaggle/working/gaussian-splatting
!python render.py -m ../output/scene_run1 --skip_train

# compute PSNR: gt in test/.../gt, prediction in test/.../renders
import numpy as np
from PIL import Image
gt_dir = Path('/kaggle/working/output/scene_run1/test')
renders = sorted(gt_dir.rglob('renders/*.png'))
gts = sorted(gt_dir.rglob('gt/*.png'))
def psnr(a, b):
    mse = np.mean((a.astype(float)-b.astype(float))**2)
    return 100 if mse == 0 else 20*np.log10(255.0/np.sqrt(mse))
scores = []
for r in renders:
    g = Path(str(r).replace('renders', 'gt'))
    if not g.exists():
        continue
    a = np.array(Image.open(g).convert('RGB'))
    b = np.array(Image.open(r).convert('RGB'))
    b = b[:a.shape[0], :a.shape[1]]
    scores.append(psnr(a, b))
print(f'PSNR: {np.mean(scores):.2f} +- {np.std(scores):.2f} dB over {len(scores)} views' if scores else 'no test views rendered (did you train without --eval?)')

## Step 8: package for download

In [ ]:
%cd /kaggle/working
!tar -czf output.tar.gz output/
!ls -lh output.tar.gz
print('Download output.tar.gz from the Kaggle output tab')